In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score


In [2]:
df_churn = pd.read_csv('Churn_Modelling.csv')

In [17]:
X_churn = df_churn.drop(['RowNumber', 'CustomerId', 'Surname', 'Exited'], axis=1)
y_churn = df_churn['Exited']
X_churn = pd.get_dummies(X_churn, columns=['Geography', 'Gender'], drop_first=True)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn
)
scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)

In [20]:
rf_param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

rf = RandomForestClassifier(random_state=42)

rf_random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=50,        
    cv=5,                
    scoring='roc_auc',   
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_random_search.fit(X_train_c, y_train_c)

print(" Random Forest :")
print("Best Parameters:", rf_random_search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
 Random Forest :
Best Parameters: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': True}


In [22]:
best_rf = rf_random_search.best_estimator_
y_pred_rf = best_rf.predict(X_test_c)
y_proba_rf = best_rf.predict_proba(X_test_c)[:, 1]

print("Accuracy test:", accuracy_score(y_test_c, y_pred_rf))


Accuracy test: 0.8685
